### Task 3: Sun Pointing Reference Frame Orientation
#### Determine the DCM between sun frame and inertial frame

In [1]:
import numpy as np
from misc_functions import create_submission_txt
from reference_frames import Reference
ref_lmo = Reference('lmo')

# Defining the reference frame
ref_sun = ref_lmo.sun_pointing_ref()
print(f'RN = {ref_sun['RN']}')
print(f'omega_RN = {ref_sun['omega']}')

create_submission_txt('submission_files/m2_t3_rn.txt',ref_sun['RN'])
create_submission_txt('submission_files/m2_t3_omega_rn',ref_sun['omega'])


RN = [[-1  0  0]
 [ 0  0  1]
 [ 0  1  0]]
omega_RN = [0 0 0]


### Task 4: Nadir Pointing Reference Frame
#### Find the reference frame that points to the surface

In [2]:
from reference_frames import Reference
from misc_functions import create_submission_txt
ref_lmo = Reference('lmo')

ref_nad = ref_lmo.nadir_pointing_ref(330,True)
create_submission_txt('submission_files/m2_t4_rn330',ref_nad['RN'])
create_submission_txt('submission_files/m2_t4_w330',ref_nad['RN'].T@ref_nad['omega'])

print(ref_nad['RN'] @ -ref_nad['omega'])

[RN] at t = 330:
    [RN] = [[ 0.07258174 -0.87057754 -0.48664837]
 [-0.98259221 -0.1460794   0.11477527]
 [-0.17101007  0.46984631 -0.8660254 ]]
    r1 = [ 0.07258174 -0.87057754 -0.48664837]
    r2 = [-0.98259221 -0.1460794   0.11477527]
    r3 = [-0.17101007  0.46984631 -0.8660254 ]
    omega = [ 0.         0.        -0.0008848]
[-0.00043058  0.00010155 -0.00076626]


### Task 5
#### Relative pointing reference frame

In [3]:
import numpy as np
from reference_frames import Reference
from misc_functions import create_submission_txt,vector_extract
from plotting_functions import plot_2d
ref_lmo = Reference('lmo')

# Finding state and creating submission files
gmo_relative = ref_lmo.gmo_pointing_ref(330,dt = 1,terminal_out=True)
create_submission_txt('submission_files/m2_t5_RN330',gmo_relative['RN'])
create_submission_txt('submission_files/m2_t5_w330',gmo_relative['RN'].T @ gmo_relative['omega'])


# Calculating logarithmic convergence plots
dt_list = 10**np.linspace(-8,4,140)
omega = []
line_names = []
error1 = []
error2 = []
error3 = []

# Determining omega for each value of dt
ref_truth = ref_lmo.gmo_pointing_ref(330,dt=1e-9)
omega_truth = ref_truth['omega']

for dt in dt_list:
    gmo_relative = ref_lmo.gmo_pointing_ref(330,dt = dt)
    omega.append(gmo_relative['omega'])

# Calculating the error
for i in range(len(omega)):
    error1.append(np.log10(np.abs(omega[i][0] - omega_truth[0])))
    error2.append(np.log10(np.abs(omega[i][1] - omega_truth[1])))
    error3.append(np.log10(np.abs(omega[i][2] - omega_truth[2])))
dt_log = [np.log10(dt) for dt in dt_list]

# Creating individual lists
w1,w2,w3 = vector_extract(omega)
x_log= [dt_log,dt_log,dt_log]
x = [dt_list,dt_list,dt_list]
# Plotting
line_names = ['omega_1','omega_2','omega_3']
plot_2d(y=[error1,error2,error3],x=x_log,xlabel='log10(dt) (log(s))',ylabel='log10(error)',line_name=line_names,title='Finite Difference Convergence')
plot_2d(y=[w1,w2,w3],x=x_log,xlabel='log10(dt) (log(s))',ylabel='omega (rad/s)',line_name=line_names,title='Finite Difference Convergence')


[RN] at t = 330:
    [RN] = [[ 0.26547539  0.96092816  0.07835742]
 [-0.96389181  0.26629415  0.        ]
 [-0.02086612 -0.07552807  0.99692533]]
    r1 = [0.26547539 0.96092816 0.07835742]
    r2 = [-0.96389181  0.26629415  0.        ]
    r3 = [-0.02086612 -0.07552807  0.99692533]
    omega = [ 1.49897687e-05 -2.05240014e-05  1.90711820e-04]


#### Convergence study

In [4]:
import numpy as np
from misc_functions import un_tilde, tilde
from plotting_functions import plot_2d

# Rotation matrix
def Rz(theta):
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([
        [c,  s, 0],
        [-s, c, 0],
        [0,  0, 1]
    ])

def test_omega_convergence(t, dt, omega0=1.0):
    """
    Returns:
        omega_true
        omega_est_forward
        omega_est_central
    """

    # ---- Truth ----
    theta = omega0 * t
    R = Rz(theta)

    omega_true = np.array([0, 0, omega0])

    # ---- Forward difference ----
    theta_p = omega0 * (t + dt)
    R_p = Rz(theta_p)

    R_dot_fwd = (R_p - R) / dt
    omega_tilde_fwd = R_dot_fwd @ R.T
    omega_fwd = un_tilde(omega_tilde_fwd)

    # ---- Central difference ----
    theta_m = omega0 * (t - dt)
    R_m = Rz(theta_m)

    R_dot_ctr = (R_p - R_m) / (2 * dt)
    omega_tilde_ctr = R_dot_ctr @ R.T
    omega_ctr = un_tilde(omega_tilde_ctr)

    return omega_true, omega_fwd, omega_ctr

dt_list = 10**np.linspace(-8, -1, 50)

err_fwd = []
err_ctr = []

for dt in dt_list:
    w_true, w_fwd, w_ctr = test_omega_convergence(t=1.0, dt=dt)

    err_fwd.append((np.linalg.norm(w_fwd - w_true)))
    err_ctr.append((np.linalg.norm(w_ctr - w_true)))

dt_log = [np.log10(dt) for dt in dt_list]

x_log= [dt_log,dt_log]
x = [dt_list,dt_list,dt_list]
# Plotting
line_names = ['Forward Difference','Central Difference']
plot_2d(y=[err_fwd,err_ctr],x=x,xlabel='log10(dt)',ylabel='log10(error)',line_name=line_names,title='Finite Difference Convergence')

